## Data Acquisition: News API

In [1]:
import os
import re
import sys
import html

import pandas as pd

from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from newsapi import NewsApiClient

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

In [2]:

# ----------------------------------------
# Paths
# ----------------------------------------
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
annotators_path = os.path.join(base_data_path, "news_api", "annotators")

# ----------------------------------------
# Regex: match NewsAPI truncation artifact
# e.g. "… [+8225 chars]"
# ----------------------------------------
TRUNC_PATTERN = re.compile(r"\u2026\s+\[\+\d+\s+chars\]")

def remove_newsapi_truncation(text):
    if isinstance(text, str):
        return TRUNC_PATTERN.sub("", text).strip()
    return text

# ----------------------------------------
# Process all CSV predictions files
# ----------------------------------------
for filename in os.listdir(annotators_path):
    if not filename.endswith(".csv"):
        continue

    filepath = os.path.join(annotators_path, filename)
    print(f"Cleaning: {filename}")

    df = DataProcessing.load_from_file(filepath, file_type="csv")

    # Only clean text-bearing columns if present
    for col in ["Base Sentence", "title", "description", "content"]:
        if col in df.columns:
            df[col] = df[col].apply(remove_newsapi_truncation)

    # Save versioned cleaned file
    clean_name = filename.replace(".csv", "-clean.csv")
    clean_path = os.path.join(annotators_path, clean_name)

    df.to_csv(clean_path, index=False, encoding="utf-8")
    print(f"Saved -> {clean_name}")

Cleaning: earnings_revenue_inflation_interest_rates_expected_forecast_outlook_projected_2026-03-21_to_2026-03-28_predictions-v3.csv
Saved -> earnings_revenue_inflation_interest_rates_expected_forecast_outlook_projected_2026-03-21_to_2026-03-28_predictions-v3-clean.csv
Cleaning: tesla_sae_level_4_autonomy_2026-03-21_to_2026-03-28_predictions-v2.csv
Saved -> tesla_sae_level_4_autonomy_2026-03-21_to_2026-03-28_predictions-v2-clean.csv
Cleaning: disease_virus_vaccine_public_health_expected_forecast_projected_prognosis_2026-03-21_to_2026-03-28_predictions-v1.csv
Saved -> disease_virus_vaccine_public_health_expected_forecast_projected_prognosis_2026-03-21_to_2026-03-28_predictions-v1-clean.csv
